> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 5 – Pré-processamento I: corte técnico e Savitzky-Golay

**Objetivo:** tratar o espectro como matriz amostras × bandas e operar ao longo de `axis=1`.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import carregar_dominio, preencher_nan, subamostra, NIR_FILE
from scipy.signal import savgol_filter

In [ ]:
meta, X, wl = carregar_dominio(NIR_FILE)
X = preencher_nan(X)
meta, X = subamostra(meta, X, n=500)   # 500 espectros para rodar rápido
print(X.shape)

## Exercício 1 – Corte técnico
Mantenha só as bandas com comprimento de onda `>= minimo`. Use uma máscara booleana.

In [ ]:
def cortar(X, wl, minimo):
    m = wl >= minimo
    return X[:, m], wl[m]

X_c, wl_c = cortar(X, wl, 450)
print(X.shape, '->', X_c.shape, '| início:', wl_c.min())

## Exercício 2 – Savitzky-Golay
Use `savgol_filter` ao longo de `axis=1`. Calcule `delta` como a mediana de `np.diff(wl)`.

In [ ]:
def suavizar(X, wl, w=11, p=2, deriv=0):
    delta = float(np.median(np.abs(np.diff(wl))))
    return savgol_filter(X, w, p, deriv=deriv, delta=delta, axis=1)

## Exercício 3 – Comparar SG0, SG1 e SG2 da mesma amostra

In [ ]:
i = 0
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for d, a in zip((0, 1, 2), ax):
    a.plot(wl_c, suavizar(X_c, wl_c, deriv=d)[i])
    a.set_title(f'SG{d}'); a.grid(alpha=0.2)
ax[-1].set_xlabel('nm')
fig.tight_layout(); plt.show()

## Exercício 4 – Efeito do tamanho da janela
Compare janelas 5, 11 e 31 na 1ª derivada, numa região estreita (2100–2300 nm).

In [ ]:
m = (wl_c >= 2100) & (wl_c <= 2300)
for w in (5, 11, 31):
    plt.plot(wl_c[m], suavizar(X_c, wl_c, w=w, deriv=1)[0, m], label=f'janela {w}')
plt.legend(); plt.xlabel('nm'); plt.title('SG1 com janelas diferentes'); plt.show()

## Checkpoint

Por que a janela precisa ser ímpar e maior que a ordem do polinômio? O que acontece com uma banda de absorção estreita quando a janela é grande demais?